# 🦅 PREDATOR Analytics v56.4.3-AUTONOMOUS — SOVEREIGN MASTER

Цей блокнот налаштований на **повну автономну роботу**. Всі ключі та конфігурації інтегровані.

### ⚡ Параметри кластера:
1. **Backend URL**: `https://predator-api.share.zrok.io` (зарезервовано)
2. **API Status**: OSINT Failover Mode
3. **Token Status**: 1eeje4um7yvA (Enabled)

> **⚠️ СТАТУС**: ПОВНА АВТОНОМІЯ. Просто запустіть 'Run all'.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
print("✅ GDrive змонтовано для синхронізації.")

In [ ]:
%%bash
# 1. Системна підготовка та 8 DB
echo "📦 Встановлення Predator OSINT Stack..."
apt-get update -qq && apt-get install -y -qq postgresql-16 redis-server openjdk-17-jdk curl wget jq 2>/dev/null
service postgresql start
service redis-server start
su -c "psql -c \"CREATE USER predator WITH PASSWORD 'predator2026' SUPERUSER;\"" postgres 2>/dev/null || true
su -c "psql -c \"CREATE DATABASE predator_db OWNER predator;\"" postgres 2>/dev/null || true

# ZROK Installation
wget -q https://github.com/openziti/zrok/releases/download/v0.4.45/zrok_0.4.45_linux_amd64.tar.gz -O /tmp/zrok.tar.gz
tar -xzf /tmp/zrok.tar.gz --wildcards '*/zrok' -O > /usr/local/bin/zrok
chmod +x /usr/local/bin/zrok
zrok disable 2>/dev/null || true
zrok enable 1eeje4um7yvA
echo "✅ Бази та ZROK готові."

In [ ]:
%%bash
# 2. FastAPI Sovereign Backend
pip install -q fastapi 'uvicorn[standard]' psycopg2-binary redis httpx
mkdir -p /opt/predator-api
cat > /opt/predator-api/main.py << 'PYEOF'
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from datetime import datetime

app = FastAPI(title="PREDATOR Sovereign API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

@app.get("/api/v1/azr/status")
def status(): return {"status": "operational", "mode": "sovereign", "node": "colab-autonomous"}

@app.get("/health")
def health(): return {"status": "ok", "ts": datetime.now().isoformat()}

@app.post("/api/v1/auth/login")
def login(c: dict): return {"access_token": "auton_master_token", "token_type": "bearer", "user": {"username": "admin"}}
PYEOF

pkill -f uvicorn || true
nohup python3 -m uvicorn main:app --host 0.0.0.0 --port 8000 --workers 4 > /var/log/predator-api.log 2>&1 &
echo "✅ API вузол запущено."

In [ ]:
import subprocess, time, httpx
print("📡 Перевірка готовності...")
for _ in range(10):
    try:
        if httpx.get("http://localhost:8000/health").status_code == 200: break
    except: time.sleep(2)

print("🌐 Резервування та запуск тунелю (predator-api)...")
subprocess.run("pkill -f zrok", shell=True)
# Спроба резервування (ігноруємо помилку, якщо вже зарезервовано)
subprocess.run(["zrok", "reserve", "public", "localhost:8000", "--name", "predator-api"], capture_output=True)
# Запуск
subprocess.Popen(["zrok", "share", "reserved", "predator-api", "--headless"])
time.sleep(5)
print("🚀 АВТОНОМНИЙ КЛАСТЕР ЗАПУЩЕНО: https://predator-api.share.zrok.io")